In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
complaints = pd.read_csv('../data/complaints_full/complaints.csv')

/var/folders/4y/0t9f134178bbfj5m299wl_7h0000gn/T/ipykernel_10038/426803039.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  complaints = pd.read_csv('../data/complaints_full/complaints.csv')


In [9]:
complaints.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2024-11-04T18:11:36.000Z,Mortgage,FHA mortgage,Closing on a mortgage,Changes in loan terms during or after closing,NaN,NaN,"Better Mortgage, Inc.",MD,21117,NaN,Phone,2024-11-04T19:12:35.000Z,Closed with explanation,No,10684182
1,2025-03-02T12:43:51.000Z,Student loan,Federal student loan servicing,Struggling to repay your loan,"Problem with forgiveness, cancellation, or dis...",This school XXXX XXXXXXXX XXXX XXXX XXXX was u...,NaN,MOHELA,FL,33309,NaN,Web,2025-04-07T16:59:46.000Z,Closed with explanation,No,12285693
2,2025-03-23T12:17:45.000Z,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Received bad information about your loan,I was placed in processing forbearance by XXXX...,NaN,MOHELA,CT,060XX,NaN,Web,2025-04-07T15:15:04.000Z,Closed with explanation,No,12661903
3,2025-04-07T21:53:35.000Z,Student loan,Federal student loan servicing,Dealing with your lender or servicer,Received bad information about your loan,Type of Issue : Loan balance dispute Loan Serv...,NaN,MOHELA,MI,48307,NaN,Web,2025-04-07T22:04:49.000Z,Closed with explanation,No,12868665
4,2025-04-13T22:18:56.000Z,Student loan,Federal student loan servicing,Struggling to repay your loan,Problem with your payment plan,NaN,NaN,MOHELA,VA,22030,NaN,Web,2025-04-13T22:31:23.000Z,Closed with explanation,No,12966383


In [12]:
# drop rows and create simple model based on email content
complaint_timely_response = complaints[['Consumer complaint narrative', 'Timely response?']].dropna()

# Convert target to binary
complaint_timely_response['label'] = (complaint_timely_response['Timely response?'] == 'Yes').astype(int)

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    complaint_timely_response['Consumer complaint narrative'],
    complaint_timely_response['label'],
    test_size=0.2,
    random_state=42
)

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, stop_words='english')),
                ('clf', LogisticRegression(max_iter=1000))])

In [15]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

              precision    recall  f1-score   support

           0       0.45      0.01      0.02      8325
           1       0.99      1.00      0.99    752543

    accuracy                           0.99    760868
   macro avg       0.72      0.50      0.51    760868
weighted avg       0.98      0.99      0.98    760868

ROC-AUC: 0.872


In [20]:
sample = ["okay"]
prob = model.predict_proba(sample)[:, 1][0]
print(f"Probability of timely response: {prob:.2%}")

Probability of timely response: 98.95%
